# Develop and test a QEC protocol (qodec)

Start with a code's stabilizers and logical operators, then turn them into a protocol we can inspect and test. The first build audits cleanly, but some circuits lose the code's distance. We will deliberately remove equations to explore audit diagnostics, derive the missing equations, then improve the circuits while keeping all six supported instructions.

`qodec` holds the code, instruction definitions, circuits, and measurement equations. `qdk.ec` builds circuits and checks their consequences. Our endpoint is a qodec with **no audit diagnostics** and **no undetected single-fault logical failure under the fault model below**. These are separate checks: correct noiseless behavior does not guarantee fault tolerance.

## Install

```bash
pip install "qdk[ec]"
```

## The desired result (TODO)
 TODO: Show a plot or some other metrics of a working 4.2.2 protocol.

## 1. Define your code

The $[[4,2,2]]$ code stores two logical qubits in four physical qubits. Its two stabilizers are $XXXX$ and $ZZZZ$. The `x` and `z` lists below choose the logical operators and their order.

In [40]:
import qodec

c4 = qodec.Code(
    "C4",
    stabilizers=["X_0 X_1 X_2 X_3", "Z_0 Z_1 Z_2 Z_3"],
    x=["X_0 X_1", "X_0 X_2"],
    z=["Z_0 Z_2", "Z_0 Z_1"],
)

### Check the code distance

`CodeProfile` supplies algebraic analysis of the code definition. Its distance search returns a `Distance` with a witness: selected single-qubit Pauli errors that preserve the stabilizers but change the encoded information. `distance.witness.product` gives their combined Pauli directly; `distance.witness.factors` retains the selected errors.

In [41]:
import qdk.ec as ec

code = ec.CodeProfile(c4)
code_distance = code.distance()
logical_error = code_distance.witness.product

print("Code distance:", code_distance)
print("Witness:", logical_error)
print("Syndrome:", sorted(code.syndrome_of(logical_error)))
print("Logical effect:", code.logical_effect_of(logical_error))
assert code_distance == logical_error.weight == 2
assert not code.syndrome_of(logical_error)
assert code.logical_effect_of(logical_error).weight > 0

Code distance: 2
Witness: XX
Syndrome: []
Logical effect: X


The code detects any single-qubit data error, but it cannot correct every such error without extra information. A circuit fault can spread to several data qubits, so the code distance does not certify the gadgets.

## 2. Build a baseline qodec

Build a qodec directly from this definition with `strategy="bare-css/v1"`. This construction uses syndrome ancillas but **no flag qubits**, and is not fault tolerant. `strict=False` keeps candidates that pass completion and logical-action verification, and records the ones it cannot build. Every returned qodec must still pass a clean audit.

A qodec is an ordered stack of layers. Each layer has an instruction set; its gadgets implement those instructions using circuits over the next layer. This build has one encoded C4 layer and one physical `stim` layer.

An instruction describes the logical operation. A gadget adds its circuit, input/output encodings, checks, and logical readouts. One encoded block contains four physical qubits here; `transversal_cx` takes two blocks.

Inspect the instruction menu and the omission record. Transversal H fails for this logical basis because it does not act as H on each logical qubit. It is absent from both the instruction set and the gadgets. Logical Paulis are frame updates, not circuit candidates.

`logical_layer` and `gadgets` refer to the contents of `protocol`, not copies. Edits through them change `protocol` immediately. We keep this one protocol throughout the walkthrough.

In [42]:
from IPython.display import display

protocol = ec.build_qodec(c4, strategy="bare-css/v1", strict=False)
logical_layer = protocol.layers[0]
gadgets = logical_layer.gadgets
instruction_menu = set(logical_layer.instruction_set.instructions)
assert instruction_menu == set(gadgets) == {
    "prepare_x", "prepare_z", "idle", "measure_x", "measure_z", "transversal_cx"
}
print(protocol.dumps()[:800])

---
qodec.yaml:
  name: C4
  description: 'Built from the ''C4'' stabilizer code ([[4, 2]]). Strategy: bare-css/v1.'
  layers:
  - instruction_set: C4.isa.yaml
    codes:
      C4: C4.code.yaml
    gadgets:
      idle: idle.gadget.yaml
      measure_x: measure_x.gadget.yaml
      measure_z: measure_z.gadget.yaml
      prepare_x: prepare_x.gadget.yaml
      prepare_z: prepare_z.gadget.yaml
      transversal_cx: transversal_cx.gadget.yaml
  - instruction_set: stim.isa.yaml
  metadata:
    qdk.ec:
      build:
        code: C4
        flags_per_stabilizer: 0
        logical_qubits: 2
        omitted:
          transversal_h:
            kind: ActionMismatch
            message: logical action differs between declared and realized
            stage: verification
        physical_qubits: 4
    


### Inspect the idle gadget

`idle` should preserve both logical qubits while measuring the two stabilizers. Its input and output encodings connect the logical block to circuit qubits 0-3; qubits 4 and 5 are syndrome ancillas.

Inspect the instruction and gadget directly below. Their YAML displays show the declarations without running analysis. The gadget snippet relies on its containing layer for the instruction-set and code bindings; it is not a self-contained bundle.

Checks are parity equations expected to be zero without faults. They can refer to circuit measurement bits and to encoding signs at either boundary. `GadgetProfile.objective` describes the declared instruction; `action` describes what the circuit actually does.

In [43]:
gadgets["idle"]

circuit:
  source: |
    R 4
    H 4
    CX 4 0
    CX 4 1
    CX 4 2
    CX 4 3
    H 4
    R 5
    H 5
    CZ 5 0
    CZ 5 1
    CZ 5 2
    CZ 5 3
    H 5
    M 4 5
  format: stim
  in:
    '0': qubit
    '1': qubit
    '2': qubit
    '3': qubit
  out:
    '0': qubit
    '1': qubit
    '2': qubit
    '3': qubit
in:
- C4: [0, 1, 2, 3]
out:
- C4: [0, 1, 2, 3]
checks:
- ['circuit.readouts[0]', 'in[0].stabilizers[0]']
- ['circuit.readouts[1]', 'in[0].stabilizers[1]']
- ['circuit.readouts[0]', 'out[0].stabilizers[0]']
- ['circuit.readouts[1]', 'out[0].stabilizers[1]']


In [44]:
bare_idle_profile = ec.GadgetProfile(gadgets["idle"])
objective = bare_idle_profile.objective
assert objective is not None
assert bare_idle_profile.action.is_equivalent_to(objective)

In [45]:
print(bare_idle_profile.action)

observables: FrameGroup(generators=())
stabilizers: FrameGroup(generators=())
mapping: {X: X, Z: Z, IX: IX, IZ: IZ}


## 3. Check the qodec for correctness

Run `ec.audit` on the complete qodec. The builder's circuits and declarations agree, so this baseline has no diagnostics. Require an empty `diagnostics` collection rather than just `report.ok`, which allows warnings.

Next, deliberately damage `protocol`: clear a measurement gadget's readouts, then clear a CNOT gadget's checks. Use `ec.filled` to reconstruct the missing equations before the fault-tolerance analysis.

This protocol has no source-file locations, so diagnostics identify the affected layer, gadget, and field.

In [46]:
initial_report = ec.audit(protocol)
print(initial_report)
assert not initial_report.diagnostics

audit: ok (no diagnostics)


### Remove the measurement readouts

The `measure_x` gadget promises two logical measurement results. Its `readouts` list supplies an equation for each result, specifying which physical measurement bits and frame signs to combine.

Clear that list in place. The circuit still measures the qubits, but the gadget no longer says how to obtain either logical result. The audit reports two `gadget/missing-observable` errors, one for each missing equation.

In [47]:
gadgets["measure_x"].readouts.clear()
print(ec.audit(protocol))

[ERROR] gadget/missing-observable
layers[0].gadgets['measure_x'] (C4 -> stim)
readouts[0] has no equation for the required logical X_0 measurement
    Expected: 2 observable bindings; declared: 0.

[ERROR] gadget/missing-observable
layers[0].gadgets['measure_x'] (C4 -> stim)
readouts[1] has no equation for the required logical X_1 measurement
    Expected: 2 observable bindings; declared: 0.

audit: 2 error(s), 0 warning(s), 0 informational


### Remove the CNOT checks

The `transversal_cx` gadget's `checks` list contains four equations relating input and output stabilizer signs. These equations describe the circuit; they do not add gates or measurements.

Clear that list in place. The circuit still implements the right logical CNOT, but its declarations no longer determine the four output stabilizer signs. The audit reports four `gadget/incomplete-output-frame` warnings in addition to the two missing-readout errors.

Fill in the equations for only the two damaged gadgets, then assign the returned gadgets to the live `gadgets` mapping. `ec.filled` returns new gadgets with reconstructed equations, including incoming logical-frame signs and output stabilizer relations. The protocol and its layers stay in place. Require a clean audit before proceeding.

In [48]:
gadgets["transversal_cx"].checks.clear()
print(ec.audit(protocol))

[ERROR] gadget/missing-observable
layers[0].gadgets['measure_x'] (C4 -> stim)
readouts[0] has no equation for the required logical X_0 measurement
    Expected: 2 observable bindings; declared: 0.

[ERROR] gadget/missing-observable
layers[0].gadgets['measure_x'] (C4 -> stim)
readouts[1] has no equation for the required logical X_1 measurement
    Expected: 2 observable bindings; declared: 0.

[WARNING] gadget/incomplete-output-frame
layers[0].gadgets['transversal_cx'] (C4 -> stim)
Declared relations do not determine out[0].stabilizers[0] (X_0 X_1 X_2 X_3)
    Code: C4; circuit support: ['0', '1', '2', '3'].
    Verified relation: ["out[0].stabilizers[0]", "in[0].stabilizers[0]", "in[1].stabilizers[0]"]

[WARNING] gadget/incomplete-output-frame
layers[0].gadgets['transversal_cx'] (C4 -> stim)
Declared relations do not determine out[0].stabilizers[1] (Z_0 Z_1 Z_2 Z_3)
    Code: C4; circuit support: ['0', '1', '2', '3'].
    Verified relation: ["out[0].stabilizers[1]", "in[0].stabilizers[

In [ ]:
for mnemonic in ("measure_x", "transversal_cx"):
    gadgets[mnemonic] = ec.filled(gadgets[mnemonic])

report = ec.audit(protocol)
print(report)
assert not report.diagnostics

audit: ok (no diagnostics)


## 4. Check the gadgets for fault tolerance

The declarations now pass every audit rule, including the output-frame checks. That establishes noiseless correctness under the supported model, not resistance to faults.

**Code distance counts data errors; gadget distance counts circuit faults.** `GadgetProfile.distance()` finds the fewest allowed faults that leave all declared checks and flags zero, preserve the output codespaces, and change the logical action. It returns a `Distance` whose witness retains both the selected fault factors and their combined product.

The default model combines post-call Pauli errors with flips of the readout bits produced by that call. Each event costs one, even when several qubits and readouts are affected together. Calls without readouts have three one-qubit or fifteen two-qubit Pauli errors.

A readout flip changes the recorded result without changing the surviving quantum state. For a non-destructive Pauli measurement, this is equivalent to an anticommuting Pauli before and after the measurement. `ec.FaultEvent.after(7, readout_flips=0)` flips the first readout produced by call 7; `[0, 2]` would select its first and third readouts. The same constructor accepts a Pauli error, a readout change, or both.

Run the exact search without a cutoff for every gadget. Each distance below has a replayable witness.

In [50]:
from IPython.display import Markdown


def report_gadget_distances(protocol: qodec.Qodec):
    distances = {
        mnemonic: ec.GadgetProfile(gadget).distance()
        for mnemonic, gadget in protocol.layers[0].gadgets.items()
    }
    rows = [
        "| Gadget | Distance | Witness |",
        "| --- | ---: | --- |",
    ]
    rows.extend(
        f"| `{mnemonic}` | {distance} | {distance.witness} |"
        for mnemonic, distance in sorted(distances.items())
    )
    display(Markdown("\n".join(rows)))


report_gadget_distances(protocol)

| Gadget | Distance | Witness |
| --- | ---: | --- |
| `idle` | 1 | X_4 after call 3 |
| `measure_x` | 2 | X_0 after call 0; X_1 after call 1 |
| `measure_z` | 2 | flip call 0 readout 0; flip call 1 readout 0 |
| `prepare_x` | 1 | X_5 after call 18 |
| `prepare_z` | 1 | X_4 after call 7 |
| `transversal_cx` | 2 | X_4 after call 0; X_5 after call 1 |

## 5. Improve the gadgets

### Improve the idle gadget

Use the self-checking C4 syndrome circuit from [Reichardt, page 4, Sec. II.2](https://arxiv.org/pdf/1804.06995#page=4). The X- and Z-syndrome couplings are interleaved so each ancilla detects dangerous faults from the other. The circuit uses eight CNOTs and the same two ancillas, with no additional flag qubit.

The paper's data qubits 1-4 become 0-3 here. Ancilla 4 measures the X stabilizer and ancilla 5 the Z stabilizer. We assume these couplings are available and omit the paper's geometric swaps. The model includes quantum errors and readout flips at the listed calls, but no additional movement or idle locations.

Edit the existing circuit's `source`, clear its old checks, and use `ec.filled` to compute equations for the new gate order. Assign the returned gadget to `gadgets["idle"]`; the live mapping updates `protocol` immediately. Then compare its action with the original objective and check its distance.

In [51]:
import stim

stim.Circuit(gadgets["idle"].circuit.source).diagram()

q0: -----X-----------@-------------------------
         |           |
q1: -----|-X---------|-@-----------------------
         | |         | |
q2: -----|-|-X-------|-|-@---------------------
         | | |       | | |
q3: -----|-|-|-X-----|-|-|-@-------------------
         | | | |     | | | |
q4: -R-H-@-@-@-@-H---|-|-|-|-M:rec[0]----------
                     | | | |
q5: -------------R-H-@-@-@-@-H--------M:rec[1]-

In [52]:
reichardt_source = """R 4 5
H 4
CX 4 0
CX 2 5
CX 0 5
CX 1 5
CX 4 2
CX 4 3
CX 4 1
CX 3 5
H 4
M 4 5
"""
stim.Circuit(reichardt_source).diagram()

q0: -----X---@----------------------
         |   |
q1: -----|---|-@-----X--------------
         |   | |     |
q2: -----|-@-|-|-X---|--------------
         | | | | |   |
q3: -----|-|-|-|-|-X-|-@------------
         | | | | | | | |
q4: -R-H-@-|-|-|-@-@-@-|-H-M:rec[0]-
           | | |       |
q5: -R-----X-X-X-------X---M:rec[1]-

In [ ]:
gadgets["idle"].circuit.source = reichardt_source
gadgets["idle"].checks.clear()
gadgets["idle"] = ec.filled(gadgets["idle"])

idle_profile = ec.GadgetProfile(gadgets["idle"])
repaired_idle_distance = idle_profile.distance()
assert idle_profile.action.is_equivalent_to(bare_idle_profile.action)
assert repaired_idle_distance == code_distance
print("Repaired idle distance:", repaired_idle_distance)
print(ec.audit(protocol))

Repaired idle distance: 2
audit: ok (no diagnostics)


### Repair both preparations with a flag

Prepare `(0000 + 1111) / sqrt(2)` on data qubits 0-3 using H on qubit 0 and CNOTs from qubit 0 to the other three. This is logical `|00>` for our C4 basis. Applying H to all four data qubits gives logical `|++>`.

Reset flag qubit 4 to zero and bracket the three data CNOTs with two `CX 0 4` gates. Without faults, their effects on the flag cancel. Measure qubit 4 and publish its result as `reject`. Each preparation uses five CNOTs and one flag qubit, with no appended syndrome extraction.

For example, X on qubit 0 after `CX 0 2` spreads to `X_0 X_3` and flips the flag. The data error preserves the codespace but changes the prepared logical state. In `prepare_x`, the final H gates turn it into `Z_0 Z_3`. Hiding the flag with a recorded-bit flip costs a second fault.

Set each instruction's `flags` through `gadget.implements`. This is the same mutable instruction held by `logical_layer.instruction_set`, so no replacement instruction is needed. Edit the circuit source and flag binding, clear the old checks, then derive the gadget's equations. Compute the distances and audit the protocol. `distance()` requires all checks and flags to remain zero for the combined fault. The gadget reports the flag; the caller decides whether to discard the result.

In [ ]:
flagged_preparation_source = """R 0 1 2 3 4
H 0
CX 0 4
CX 0 1
CX 0 2
CX 0 3
CX 0 4
"""
preparation_suffixes = {
    "prepare_z": "M 4\n",
    "prepare_x": "H 0 1 2 3\nM 4\n",
}
for mnemonic, suffix in preparation_suffixes.items():
    gadget = gadgets[mnemonic]
    gadget.implements.flags = ["reject"]
    gadget.circuit.source = flagged_preparation_source + suffix
    gadget.checks.clear()
    gadget.readouts = [{"reject": ["circuit.readouts[0]"]}]
    gadgets[mnemonic] = ec.filled(gadget)
    print(mnemonic, "distance:", ec.GadgetProfile(gadgets[mnemonic]).distance())
print(ec.audit(protocol))

prepare_z distance: 2
prepare_x distance: 2
audit: ok (no diagnostics)


### How the idle ancillas check each other

The gate order makes a dangerous ancilla fault visible to the other ancilla:

- **X on ancilla 4 after `CX 4 2`:** the remaining couplings spread X to data qubits 3 and 1. Qubit 3 still has to couple to ancilla 5 through `CX 3 5`, so it flips the Z-syndrome result. Qubit 1 has already coupled to ancilla 5, so its X cannot cancel that flip.
- **Z on ancilla 5 after `CX 0 5`:** `CX 1 5` spreads Z backwards to data qubit 1, and the later `CX 4 1` carries it to ancilla 4. The final H turns that Z into X, flipping the X-syndrome result. The Z that reaches data qubit 3 arrives after `CX 4 3`, so it cannot cancel the ancilla error.

These are separate experiments, evaluated together by `effects_of`. In `Circuit.calls`, the injection gates are at positions 7 and 5. The first two assertions verify those positions before creating the faults.

The X-syndrome bit appears in checks 0 and 2; the Z-syndrome bit appears in checks 1 and 3. Each pair compares the same measurement with the input and output stabilizer signs. Thus `[1, 3]` means the Z-syndrome bit reveals the X fault, not that two independent measurements detected it.

These examples explain the idle circuit. The preparations use the separate single-flag circuits above. Their distance searches include quantum and readout errors and require all checks and flags to remain zero.

In [55]:
idle_calls = gadgets["idle"].circuit.calls()
assert (idle_calls[7].mnemonic, idle_calls[7].operands) == ("CX", [4, 2])
assert (idle_calls[5].mnemonic, idle_calls[5].operands) == ("CX", [0, 5])

x_hook = ec.FaultEvent.after(7, ec.Pauli("X_4"))
z_hook = ec.FaultEvent.after(5, ec.Pauli("Z_5"))
x_effect, z_effect = idle_profile.effects_of([x_hook, z_hook])

print("X on ancilla 4 is detected by Z checks:", sorted(x_effect.syndrome))
print("Z on ancilla 5 is detected by X checks:", sorted(z_effect.syndrome))
assert x_effect.syndrome == {1, 3}
assert z_effect.syndrome == {0, 2}

X on ancilla 4 is detected by Z checks: [1, 3]
Z on ancilla 5 is detected by X checks: [0, 2]


## 7. Check the gadgets (mark 2) for fault tolerance

Re-run the audit and every gadget's distance search after the edits. The live `gadgets` mapping still refers to the same protocol's first layer.

The final menu still has both preparations, both destructive measurements, idle, and transversal CNOT. Both preparation instructions now return a `reject` flag. All six gadgets have distance two under the quantum and readout-fault model, with a two-event witness for each and all checks and flags zero.

Require **no diagnostics**, rather than only `report.ok`: the latter allows warnings. These checks establish noiseless consistency and the stated gadget-distance property separately.

In [56]:
protocol.description = (
    "C4 built with bare-css/v1, with self-checking idle "
    "and single-flag preparation circuits."
)
print(ec.audit(protocol))
report_gadget_distances(protocol);

audit: ok (no diagnostics)


| Gadget | Distance | Witness |
| --- | ---: | --- |
| `idle` | 2 | X_4 after call 0; Z_5 after call 5 |
| `measure_x` | 2 | X_0 after call 0; X_1 after call 1 |
| `measure_z` | 2 | flip call 0 readout 0; flip call 1 readout 0 |
| `prepare_x` | 2 | X_1 after call 1; X_2 after call 2 |
| `prepare_z` | 2 | X_1 after call 1; X_2 after call 2 |
| `transversal_cx` | 2 | X_4 after call 0; X_5 after call 1 |

### Save the qodec

A qodec carries the revised circuits and equations as ordinary data. `save` takes a directory and returns the written file's path. With `single_file=True`, that file contains the whole bundle.

Use a temporary directory to check the round trip. Load the saved bundle only for the equality check; keep working with `protocol`. Serialization preserves a design; it does not prove the design correct.

In [57]:
from tempfile import TemporaryDirectory

with TemporaryDirectory() as directory:
    saved_path = protocol.save(directory, single_file=True)
    assert qodec.Qodec.load(saved_path) == protocol
print("Round trip: OK")

Round trip: OK


## 8. Evaluate performance (TODO)

## What we established

Starting only from the C4 code, we built an audit-clean bare qodec and inspected its six retained instructions. Deleting readout equations and output stabilizer relations demonstrated the audit diagnostics; `ec.filled` reconstructed those declarations and restored a clean audit. We edited the same protocol to give idle self-checking syndrome extraction and both preparations five-CNOT circuits returning a `reject` flag.

The final `protocol` has no audit errors, warnings, or informational findings. All six gadgets have distance two for faults that leave all checks and flags zero, including both destructive measurements, with quantum and recorded-bit errors included. The save/load check preserves the complete declarations.

This is a gadget-level result under the stated Pauli and readout-fault model, not a certificate for every hardware implementation or composition. Before using the protocol with a device, account for its connectivity, timing, additional fault locations, and noise correlations.